# Predict — Dhaka twin forecasting on Colab (GPU)

Trains the official BasicTS-legacy baselines (STID, GWNet, DCRNN, AGCRN, STAEformer, STGCN) on the Dhaka twin dataset
(`predict/data/dhaka_twin`, label DHAKA_SIMULATED_SENSOR_GRAPH). Dataset and results live on Drive under `dhakaflow/predict/`.
Runtime → GPU. Each model takes 1–5 min on an A100/L4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, shutil, subprocess, pathlib, time
DRIVE = pathlib.Path('/content/drive/MyDrive/dhakaflow/predict')
TWIN  = DRIVE / 'data/dhaka_twin/basicts_legacy'      # copied from the repo with rclone
OUT   = DRIVE / 'results/twin_models'; OUT.mkdir(parents=True, exist_ok=True)
assert (TWIN/'data.dat').exists(), f'missing {TWIN} — rclone copy predict/data/dhaka_twin gdrive2tb:dhakaflow/predict/data/dhaka_twin'
!nvidia-smi -L

In [ ]:
# pinned legacy harness (official, unmodified) + the repo's config adapter
LEGACY_SHA = '4c3e2d373c647667c2123e1591f2d79c229dfcc6'
BTS = pathlib.Path('/content/BasicTS_legacy')
if not BTS.exists():
    !git clone -q https://github.com/GestaltCogTeam/BasicTS.git {BTS}
!git -C {BTS} checkout -q {LEGACY_SHA} && git -C {BTS} log --oneline -1
%pip -q install easytorch easydict
dst = BTS/'datasets/DhakaR1SensorGraph_legacy'; dst.mkdir(parents=True, exist_ok=True)
for f in ('data.dat','desc.json','adj_mx.pkl'): shutil.copy2(TWIN/f, dst/f)
print(json.loads((dst/'desc.json').read_text())['shape'])

In [ ]:
# research/forecasting/make_legacy_config.py, inlined (adapter rule: patch dataset name + node count only)
DATASET, NUM_NODES = 'DhakaR1SensorGraph_legacy', 250
def make(model):
    src = BTS/'baselines'/model/'PEMS04.py'
    t = src.read_text().replace("DATA_NAME = 'PEMS04'", f"DATA_NAME = '{DATASET}'")
    t = re.sub(r'("num_nodes"\s*:\s*)307', rf'\g<1>{NUM_NODES}', t)
    t = re.sub(r'("num_of_vertices"\s*:\s*)307', rf'\g<1>{NUM_NODES}', t)
    out = src.with_name(f'{DATASET}.py'); out.write_text(t); return out

RE_EVAL = re.compile(r'horizon\s+(\d+),\s*Test MAE:\s*([\d.]+),\s*Test MAPE:\s*([\d.]+),\s*Test RMSE:\s*([\d.]+)')
MODELS = ['STID', 'GWNet', 'DCRNN', 'AGCRN', 'STAEformer', 'STGCN']
results = json.loads((OUT/'twin_models.json').read_text()) if (OUT/'twin_models.json').exists() else {}
for m in MODELS:
    if m in results and results[m].get('rc') == 0: print('done already:', m); continue
    cfg = make(m); log = OUT/f'{m}.log'; t0 = time.time()
    with open(log, 'w') as fh:
        rc = subprocess.call(['python', 'experiments/train.py', '-c', str(cfg.relative_to(BTS)), '-g', '0'], cwd=BTS, stdout=fh, stderr=subprocess.STDOUT)
    txt = log.read_text(errors='ignore'); best = {}
    for h, mae, mape, rmse in RE_EVAL.findall(txt): best[int(h)] = dict(mae=float(mae), mape=float(mape), rmse=float(rmse))
    saved = re.findall(r'Test results saved to (\S+)', txt)
    if saved:
        p = BTS/saved[-1].rstrip('.'); p = p if p.exists() else p.parent/'test_results.npz'
        if p.exists(): shutil.copy2(p, OUT/f'{m}_test_results.npz')
    results[m] = dict(rc=rc, wall_s=round(time.time()-t0), best_test=best, config=str(cfg.relative_to(BTS)))
    (OUT/'twin_models.json').write_text(json.dumps(results, indent=1))
    print(f'[{m}] rc={rc} wall={time.time()-t0:.0f}s', {h: round(v['mae'],3) for h, v in sorted(best.items())})

In [ ]:
# table (15/30/60 min = horizons 3/6/12) — pull into the repo with: rclone copy gdrive2tb:dhakaflow/predict/results/twin_models predict/results/twin_models
lines = ['| model | MAE@15 | RMSE@15 | MAE@30 | RMSE@30 | MAE@60 | RMSE@60 | wall s |', '|---|---|---|---|---|---|---|---|']
for m, r in results.items():
    b = {int(k): v for k, v in r['best_test'].items()}
    lines.append(f'| {m} | ' + ' | '.join(f"{b[h]['mae']:.3f} | {b[h]['rmse']:.3f}" if h in b else '– | –' for h in (3,6,12)) + f" | {r['wall_s']} |")
(OUT/'twin_models_colab.md').write_text('# Dhaka twin forecasting — official BasicTS-legacy baselines (Colab GPU)\n\n' + '\n'.join(lines) + '\n')
print('\n'.join(lines))